<!-- nav -->
[← 8 · From constraints to code](08_From_Constraints_To_Code.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)

# 9 · Constraints in hardware

**The problem.** The cold chain's sensor board is an FPGA, and its team writes SystemVerilog. The constraints kept as
data with mbse-expressions, "the reading stays between 2.0 and 8.0 °C" (in tenths of a degree, as the sensor counts) and
"a shipment is fresh for 72 hours", must hold in the testbench's random stimulus and be checked in simulation. And the
RTL already checks things that nobody has written down as constraints.

Verilog is the fourth language here: Verilog and SystemVerilog as one tree language, whose kinds follow IEEE 1800's
grammar, read by slang. A second **bridge** carries mbse-expressions' SystemVerilog dialect to and from its trees, as
case study 8's did for Python.

In [1]:
from mbse.Expressions import Evaluators, Expressions as E, Translators
from mbse.Expressions.Dialects.SystemVerilog import Domains, Evaluators as SvEvaluators, Expressions as SvDialect, Text
from mbse.Programs.Bridges import Verilog as Bridge
from mbse.Programs.Framework import Errors
from mbse.Programs.Framework.Syntax import walk
from mbse.Programs.Verilog import SystemVerilog2023, Syntax as S, Verilog2005
from mbse.Schemas.Framework import Proxies, Schemas

Shipment = (Schemas.OfObject.Builder().name("Shipment").ref()
            .properties(lambda p: p.name("deci_celsius").of(lambda t: t.as_native(int)),
                        lambda p: p.name("hours").of(lambda t: t.as_native(int))).create())
store = Proxies.OfStore()
store.register(Shipment)
this = E.variable("this")
constraints = {
    "in_range": E.literal(20).le(this.deci_celsius).and_(this.deci_celsius.le(80)).data,
    "fresh": this.hours.le(72).data,
}

## Step 1: SystemVerilog as a tree

The testbench's transaction is a class of random variables. It parses into a tree like any other program, by the
standard it's written to; a class is SystemVerilog's, so Verilog-2005 refuses to print it, at its path:

In [2]:
reading = SystemVerilog2023.parse("""class reading;
    rand int deci_celsius;  // tenths of a degree
    rand int hours;
endclass
""")
print([type(item).__name__ for item in reading.items[0].items])
try:
    Verilog2005.print(reading)
except Errors.PrintError as error:
    print(error)

['VariableDeclaration', 'Comment', 'VariableDeclaration']
items[0]: ClassDeclaration is not Verilog, but this is Verilog-2005


## Step 2: constraints become `constraint` blocks

mbse-expressions translates each constraint from its neutral form (Basic) to its SystemVerilog dialect; the bridge turns
that into a `constraint` block, which joins the class's items. The randomizer then executes the specification
nondeterministically: each transaction it draws is one choice within the constraints. `this` is the constraint's
subject, and in a class it's SystemVerilog's own `this`:

In [3]:
to_sv = Translators.between(E.DIALECT, SvDialect.DIALECT)
for name, constraint in constraints.items():
    reading.items[0].items.append(Bridge.constraint(f"c_{name}", to_sv.forward(constraint)))
print(SystemVerilog2023.print(reading))

class reading;
    rand int deci_celsius;  // tenths of a degree
    rand int hours;

    constraint c_in_range {
        20 <= this.deci_celsius && this.deci_celsius <= 80;
    }

    constraint c_fresh {
        this.hours <= 72;
    }
endclass



## Step 3: constraints become checks

The same constraints become methods the testbench calls to check a transaction, and an immediate assertion that reports
where a simulation breaks them. Evaluated by the SystemVerilog dialect's rules, they agree with Basic on each shipment:

In [4]:
print(SystemVerilog2023.print(Bridge.function_("in_range", [], to_sv.forward(constraints["in_range"]))))
print(SystemVerilog2023.print(Bridge.assertion(to_sv.forward(constraints["fresh"]), "stale shipment")))
shipments = [store.Shipment().deci_celsius(55).hours(30).create(), store.Shipment().deci_celsius(91).hours(80).create()]
for shipment in shipments:
    in_hardware = [str(SvEvaluators.OfAny(to_sv.forward(constraint), {"this": shipment})) for constraint in constraints.values()]
    print(in_hardware, [Evaluators.OfAny(constraint, {"this": shipment}) for constraint in constraints.values()])

function automatic logic in_range;
    return 20 <= this.deci_celsius && this.deci_celsius <= 80;
endfunction
assert (this.hours <= 72) else $error("stale shipment");
["1'b1", "1'b1"] [True, True]
["1'b0", "1'b0"] [False, False]


## Step 4: an RTL assertion becomes a constraint

The FIFO's monitor already asserts something. The bridge reads it into a constraint, which can now be stored, reviewed
and evaluated beside the others. Evaluated by IEEE 1800's rules, it sees four states: an unknown bit makes the check
unknown (`1'bx`), as in simulation, rather than true or false.

In [5]:
monitor = SystemVerilog2023.parse("""module fifo_monitor(input logic clk, input logic [7:0] count, input logic overflow);
    always @(posedge clk) begin
        assert (count <= 8'd200 && !overflow) else $error("fifo too full");
    end
endmodule
""")
assertion = next(node for node in walk(monitor) if isinstance(node, S.ImmediateAssertion))
constraint = Bridge.term_of_assertion(assertion)
print(Text.ToText(constraint))
for count, overflow in [("00010000", "0"), ("11110000", "0"), ("00010000", "x"), ("1111xxxx", "0")]:
    value = SvEvaluators.OfAny(constraint, {"count": Domains.Logic.of(count), "overflow": Domains.Logic.of(overflow)})
    print(count, overflow, value)

count <= 8'd200 && !overflow
00010000 0 1'b1
11110000 0 1'b0
00010000 x 1'bx
1111xxxx 0 1'bx


## Step 5: what a constraint can't hold

A constraint is smaller than a design: pure, with no statements, no timing, and only the dialect's expressions. An
indexed part-select, `data[3+:2]`, has no counterpart, and the bridge says where:

In [6]:
check = SystemVerilog2023.parse("function automatic logic f(input logic [7:0] data); return data[3+:2] == 2'b01; endfunction\n")
try:
    Bridge.term_of_function(check.items[0])
except Errors.TranspileError as error:
    print(error)

body[0].value.left: a part-select with +: has no counterpart in the dialect


## Why it works this way

- **One constraint, every target.** The constraint written once as data is Python in the pipeline (case study 8), and
  here a `constraint` block the testbench's randomizer generates within and an assertion that simulation checks, each
  printed by its language's own printer. Nobody copies it by
  hand, so nobody copies it wrong.
- **Four states, not two.** The SystemVerilog dialect evaluates as IEEE 1800 does, with `x` and `z`, so a constraint
  read from RTL means there what it means in simulation.
- **Read by the reference front end.** Verilog is read by slang, in both implementations, so the trees and the
  constraints read from them are the same in Python and TypeScript.
- **Refusing is part of the contract.** What a constraint can't hold fails at a path, never approximated.

<!-- nav -->
[← 8 · From constraints to code](08_From_Constraints_To_Code.ipynb) · [Home](../../README.md) · [TypeScript tutorial →](../../typescript5/tutorials/README.md)